# DNN (v2)

In [ ]:
import numpy as np
import pandas as pd
import pickle
import sys
import os
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report

sys.path.insert(0,'..')
from src.text_features import CombinedVectorizer, encode_labels, decode_labels, CLASS_NAMES
from src.ffnn import FFNN, train_model
from src.metrics import f1_macro

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
TRAIN_PATH      = '../datasets/dataset_v2_train.csv'
TEST_PATH       = '../datasets/dataset_v2_test.csv'
VALIDATION_PATH = '../datasets/dataset_v2_val.csv'

df_train = pd.read_csv(TRAIN_PATH, sep=';')
df_test  = pd.read_csv(TEST_PATH,  sep=';')
df_val   = pd.read_csv(VALIDATION_PATH, sep=';')

print(f'treino: {len(df_train)} exemplos')
print(f'teste:  {len(df_test)}  exemplos')
print()
print('distribuição de treino:')
print(df_train['Label'].value_counts())

In [ ]:
train_texts  = df_train['Text'].fillna('').tolist()
train_labels = df_train['Label'].tolist()

test_texts   = df_test['Text'].fillna('').tolist()
test_labels  = df_test['Label'].tolist()

# ── vectorização: TF-IDF palavra (1,2)-grams + char (3,5)-grams ──────────
vec = CombinedVectorizer(max_words=10000, ngram_range=(1,2),
                         max_chars=5000,  char_range=(3,5))

X_train = vec.fit_transform(train_texts)
X_test  = vec.transform(test_texts)

Y_train = encode_labels(train_labels)
Y_test  = encode_labels(test_labels)

print(f'dimensão das features: {X_train.shape[1]}')

In [ ]:
# ── PyTorch DataLoaders ───────────────────────────────────────────────────
def make_loader(X, y, batch_size=32, shuffle=False):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, Y_train, batch_size=32, shuffle=True)
test_loader  = make_loader(X_test,  Y_test,  batch_size=64)

print(f'X_train: {X_train.shape}  Y_train: {len(Y_train)}')
print(f'X_test:  {X_test.shape}   Y_test:  {len(Y_test)}')

In [ ]:
input_dim = X_train.shape[1]
model_dnn = FFNN(input_dim=input_dim, n_classes=5,
                 topology=[512, 256, 128], dropout=0.4).to(device)

model_dnn = train_model(
    model_dnn, train_loader, test_loader,
    epochs=200, lr=0.001, patience=25, weight_decay=1e-4
)

In [ ]:
def get_metrics(model, loader):
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            out = model(X_b.to(device))
            all_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            all_true.extend(y_b.numpy())

    Y_onehot = np.zeros((len(all_true), 5))
    Y_onehot[np.arange(len(all_true)), all_true] = 1
    P_onehot = np.zeros((len(all_preds), 5))
    P_onehot[np.arange(len(all_preds)), all_preds] = 1

    from src.metrics import accuracy, f1_macro
    return accuracy(Y_onehot, P_onehot), f1_macro(Y_onehot, P_onehot), all_preds

train_loader_full = make_loader(X_train, Y_train, batch_size=64)
train_acc, train_f1, _ = get_metrics(model_dnn, train_loader_full)
test_acc,  test_f1,  _ = get_metrics(model_dnn, test_loader)

print(f'[TRAIN] accuracy: {train_acc:.4f}  f1-macro: {train_f1:.4f}')
print(f'[TEST]  accuracy: {test_acc:.4f}   f1-macro: {test_f1:.4f}')

In [ ]:
torch.save(model_dnn.state_dict(), '../models/model_dnn_v2.pt')
with open('../vectorizers/vectorizer_dnn_v2.pkl', 'wb') as f:
    pickle.dump(vec, f)

print('modelo e vectorizer guardados.')

## *validation*

In [ ]:
with open('../vectorizers/vectorizer_dnn_v2.pkl', 'rb') as f:
    vec_val = pickle.load(f)

input_dim_val = len(vec_val.word_index) + len(vec_val.char_vec.vocab)
model_val = FFNN(input_dim=input_dim_val, n_classes=5,
                 topology=[512, 256, 128], dropout=0.4).to(device)
model_val.load_state_dict(torch.load('../models/model_dnn_v2.pt', map_location=device))

val_texts  = df_val['Text'].fillna('').tolist()
val_labels = df_val['Label'].tolist()
Y_val      = encode_labels(val_labels)

X_val = vec_val.transform(val_texts)
val_loader = make_loader(X_val, Y_val, batch_size=64)

val_acc, val_f1, val_preds = get_metrics(model_val, val_loader)
print(f'accuracy: {val_acc:.4f}  f1-macro: {val_f1:.4f}')

preds_decoded = decode_labels(val_preds)
df_results = pd.DataFrame({'ID': df_val['ID'], 'Predicted': preds_decoded, 'True': df_val['Label']})
print(df_results.to_string(index=False))

## *avaliação com dataset-exemplos*

In [ ]:
EXEMPLOS_PATH = '../datasets/dataset-exemplos.csv'
df_exemplos = pd.read_csv(EXEMPLOS_PATH, sep=';')

X_ex = vec_val.transform(df_exemplos['Text'].fillna('').tolist())
Y_ex = encode_labels(df_exemplos['Label'].tolist())
ex_loader = make_loader(X_ex, Y_ex, batch_size=64)

ex_acc, ex_f1, ex_preds = get_metrics(model_val, ex_loader)
print(f'[EXEMPLOS] accuracy: {ex_acc:.4f}  f1-macro: {ex_f1:.4f}')
print()

preds_decoded_ex = decode_labels(ex_preds)
print(classification_report(df_exemplos['Label'].str.lower().tolist(), preds_decoded_ex, digits=3))

df_ex_results = pd.DataFrame({
    'ID':        df_exemplos['ID'],
    'True':      df_exemplos['Label'],
    'Predicted': [p.capitalize() for p in preds_decoded_ex]
})
df_ex_results['Correct'] = df_ex_results['True'].str.lower() == df_ex_results['Predicted'].str.lower()
print(df_ex_results.to_string(index=False))

## *avaliação com subm1 (labels reveladas)*

In [ ]:
SUBM1_PATH = '../Subm1/subm1_labels_revealed.csv'
df_subm1 = pd.read_csv(SUBM1_PATH, sep=';')

X_subm1 = vec_val.transform(df_subm1['Text'].fillna('').tolist())
Y_subm1  = encode_labels(df_subm1['Label'].tolist())
subm1_loader = make_loader(X_subm1, Y_subm1, batch_size=64)

subm1_acc, subm1_f1, subm1_preds = get_metrics(model_val, subm1_loader)
print(f'[SUBM1] accuracy: {subm1_acc:.4f}  f1-macro: {subm1_f1:.4f}')
print()

preds_decoded_subm1 = decode_labels(subm1_preds)
print(classification_report(df_subm1['Label'].str.lower().tolist(), preds_decoded_subm1, digits=3))

df_subm1_results = pd.DataFrame({
    'ID':        df_subm1['ID'],
    'True':      df_subm1['Label'],
    'Predicted': [p.capitalize() for p in preds_decoded_subm1]
})
df_subm1_results['Correct'] = df_subm1_results['True'].str.lower() == df_subm1_results['Predicted'].str.lower()
print(df_subm1_results.to_string(index=False))